In [1]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 4 – Traveling Salesman Problem
#  Section: 4.4 – TSP with MTZ constraints (TSP-MTZ)
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using JuMP    # Modeling language
using HiGHS   # Solver
using HDF5    # For reading HDF5 files

# Load utility functions for plotting
include("utils/tsp_utils.jl")

# Function to solve TSP using MTZ formulation
function solve_tsp_mtz(file_path)

    # Load the distance matrix from the HDF5 file
    D = h5read(file_path, "distance_matrix")

    # Number of locations
    n = size(D, 1)

    # Create model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Define the decision variables
    @variable(model, x[1:n, 1:n], Bin)

    # Define the MTZ variables
    @variable(model, u[2:n])

    # Objective function: minimize total distance
    @objective(model, Min, sum(D[i,j] * x[i,j] for i in 1:n, j in 1:n))

    # Constraints to eliminate self-loops
    @constraint(model, [i=1:n], x[i,i] == 0)

    # Each column sums to 1 (each location visited once)
    @constraint(model, [j=1:n], sum(x[i,j] for i in 1:n) == 1)

    # Each row sums to 1 (each location departs once)
    @constraint(model, [i=1:n], sum(x[i,j] for j in 1:n) == 1)

    # MTZ constraints
    @constraint(model, [i=2:n, j=2:n], n * x[i,j] - (n - 1) <= u[j] - u[i])

    # Run the solver
    JuMP.optimize!(model)

    # Get the values of the decision variables
    x_opt = JuMP.value.(x)

    # Extract the routes from the solution
    route = [1]
    while true
        to = findfirst(x_opt[route[end], :] .> 0.5)
        to == 1 ? break : push!(route, to)
    end
    println("Route: $route")

    # Get the optimal value of the objective function
    z_opt = JuMP.objective_value(model)
    println("Optimal value: $z_opt meters")

    # Plot the solution on the map
    p = plot_tsp_solution(file_path, x_opt);
    display(p)
end

# Example usage
solve_tsp_mtz("data/tsp_example.h5")

Route: [1, 2, 3, 5, 4]
Optimal value: 3664.7 meters


PyObject <folium.folium.Map object at 0x0000026221E2ACC0>